In [ ]:
import tensorflow as tf
import tensorflow.keras as K
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 1. ЗАГРУЗКА МОДЕЛИ
try:
    model = K.models.load_model('bluzy_bryuki_model.h5')
    print("Модель загружена!")
except OSError:
    print("ОШИБКА: Файл модели не найден. Сначала запустите обучение!")

# 2. ФУНКЦИЯ ПОДГОТОВКИ ИЗОБРАЖЕНИЯ
def prepare_image(filepath):
    # Читаем картинку
    img = cv2.imread(filepath)
    if img is None:
        print(f"Не удалось прочитать файл: {filepath}")
        return None, None
        
    # Конвертируем BGR (OpenCV) в RGB
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Изменяем размер под требования нейросети
    # ВНИМАНИЕ: cv2.resize принимает (Ширина, Высота) -> (66, 46)
    img_resized = cv2.resize(img, (66, 46))
    
    # Нормализация (0..1) и добавление размерности батча
    img_tensor = np.array(img_resized, dtype='float32') / 255.0
    img_tensor = np.expand_dims(img_tensor, axis=0)
    
    return img, img_tensor

# 3. ФУНКЦИЯ ПРЕДСКАЗАНИЯ
def predict_clothing(filepath):
    original_img, tensor_img = prepare_image(filepath)
    
    if tensor_img is not None:
        prediction = model.predict(tensor_img)
        
        class_idx = np.argmax(prediction[0])
        probability = prediction[0][class_idx]
        
        label = "НЕИЗВЕСТНО"
        if class_idx == 0:
            label = "БЛУЗКА (Bluza)"
        else:
            label = "БРЮКИ (Bryuki)"
            
        plt.figure()
        plt.imshow(original_img)
        plt.title(f"{label} ({probability*100:.1f}%)")
        plt.axis('off')
        plt.show()

# predict_clothing('images/test_bluza.jpg')
# predict_clothing('images/test_bryuki.jpg')